In [ ]:
# Step 1: Clone the repository (feature/go-phase0-pretraining branch) and install dependencies.
!git clone -b feature/go-phase0-pretraining --single-branch https://github.com/HUBioDataLab/ContVAR.git /content/ContVAR
%cd /content/ContVAR

%pip install -q graphein MDAnalysis torch_geometric torchmetrics wandb biopython h5py
!apt-get -qq install dssp
%pip install -e .

In [ ]:
# Step 2: Mount Google Drive to access data files.
# Drive is only used for two large files (not code):€
#   - protein_triplets_data_9march.zip  (CIF structure files)
#   - embeddings_variable.h5            (precomputed ESM2 embeddings)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# PATH CONFIGURATION - Adjust these paths for your environment
# ============================================================

# Google Drive root for ContVAR data
DRIVE_ROOT = "/content/drive/MyDrive/ContVAR"

# DMS data (CIF structures + split)
DATA_ROOT = "/content/content/content/protein_triplets_data"
DATA_ZIP = f"{DRIVE_ROOT}/protein_triplets_data_9march.zip"
EMBEDDINGS_PATH = f"{DRIVE_ROOT}/embeddings_variable.h5"

# Phase 0: GO semantic similarity pretraining
GO_TSV_DIR = f"{DRIVE_ROOT}/semantic_similarity"
GO_PREBUILT_GRAPH_ROOT = f"{DRIVE_ROOT}/prebuilt_graphs"
GO_ESM_H5 = f"{DRIVE_ROOT}/esm2_t33_650M_UR50D_protein_embedding.h5"

# Output paths
RESULTS_DRIVE_DIR = "/content/drive/MyDrive"

In [ ]:
# Step 3: Login to Weights & Biases and set up the environment.
import wandb
from contvar import setup_environment, train_pipeline, visualize_tsne

wandb.login(key="2becafa4dcb70173759a7b50ee5de92401c637c4")
env = setup_environment(
    data_root=DATA_ROOT,
    embeddings_path=EMBEDDINGS_PATH,
    data_zip=DATA_ZIP,
)

In [ ]:
# Step 4: Run the full training pipeline (Phase 0 + curriculum learning).
# - Phase 0: GO semantic similarity pretraining (MF/BP/CC heads) using prebuilt .pt graphs.
# - Phase 1: Exhaustive triplet training with standard triplet loss.
# - Phase 2: Streaming semi-hard negative mining with online triplet loss.

config_overrides = {
    # Phase 0 GO pretraining (set to 0 to disable)
    "go_phase0_epochs": 20,
    "go_max_triplets_per_ontology": 1000,
    "go_tsv_dir": GO_TSV_DIR,
    "go_prebuilt_graph_root": GO_PREBUILT_GRAPH_ROOT,
    "go_embeddings_path": GO_ESM_H5,

    # Ontology sampling ratio
    "go_sampling_enabled": True,
    "go_sampling_ratio": {"mf": 0.6, "bp": 0.2, "cc": 0.2},
    "go_log_sampling_stats": True,

    # Split directly from prebuilt proteins
    "go_random_split_from_prebuilt": True,
    "go_train_ratio": 0.8,
    "go_val_ratio": 0.1,
    "go_test_ratio": 0.1,
    "go_split_mode": "none",
}

model, mapper, processed_dir = train_pipeline(
    config=config_overrides,
    force=False,
    split_path=None,
    data_root=env['data_root'],
    embeddings_path=env['embeddings_path'],
    device=env['device'],
)

In [ ]:
# Step 5: Visualize learned embeddings using t-SNE.
# Plots comparing:
#   - Baseline (raw pooled node features, no projection) vs Projected (GNN output)
#   - Global (graph-level) vs Local (mutation-position) embeddings
# Each point is colored by label (benign=green, pathogenic=red, WT=gold).
save_dir = "visualizations"

visualize_tsne(
    model=model, mapper=mapper, processed_dir=processed_dir,
    device=env['device'], save_dir=save_dir, split='val', n_proteins=10,
)
visualize_tsne(
    model=model, mapper=mapper, processed_dir=processed_dir,
    device=env['device'], save_dir=save_dir, split='train', n_proteins=10,
)

In [ ]:
import os
import shutil
from datetime import datetime

timestamp = datetime.now().strftime("%d%b_%H%M")
save_dir = f'{RESULTS_DRIVE_DIR}/Protein_Model_Results_{timestamp}'
os.makedirs(save_dir, exist_ok=True)
print(f"Saving to: {save_dir}")

files_to_save = [
    'model_best_loss.pt',
    'model_last.pt',
    f'{DATA_ROOT}/split.json',
    'visualizations/tsne_comparison_val_top_10.png',
    'visualizations/tsne_comparison_train_top_10.png',
    'visualizations/tsne_per_protein_val_top_10.png',
    'visualizations/tsne_per_protein_train_top_10.png',
]

for f in files_to_save:
    if os.path.exists(f):
        shutil.copy(f, save_dir)
        print(f"Saved {f}")
    else:
        print(f"File not found: {f}")

print("All files safely transferred to Google Drive.")

In [ ]:
from google.colab import files

zip_name = f'/content/Protein_Model_Results_{timestamp}.zip'
!zip -r "{zip_name}" "{save_dir}"
files.download(zip_name)

In [ ]:
#from google.colab import runtime

#import time

#time.sleep(5)
# This will disconnect and delete the runtime
#runtime.unassign()